### Confused about how a2c rnn is implemented in jax, want to test it

In [1]:
%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp
import flax.linen as nn
import numpy as np
import optax
from flax.linen.initializers import constant, orthogonal
from typing import Sequence, NamedTuple, Any, Dict
from flax.training.train_state import TrainState
import distrax

import jaxmarl
from jaxmarl.wrappers.baselines import MPELogWrapper
from jaxmarl.environments import spaces

import functools

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
2026-02-10 17:45:33.649866: E external/xla/xla/stream_executor/cuda/cuda_driver.cc:276] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
CUDA backend failed to initialize: FAILED_PRECONDITION: No visible GPU devices. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


In [2]:


class ScannedRNN(nn.Module):
    @functools.partial(
        nn.scan,
        variable_broadcast="params",
        in_axes=0,
        out_axes=0,
        split_rngs={"params": False},
    )
    @nn.compact
    def __call__(self, carry, x):
        """Applies the module."""
        rnn_state = carry
        # ins, resets = x
        ins = x
        # rnn_state = jnp.where(
        #     resets[:, np.newaxis],
        #     self.initialize_carry(*rnn_state.shape),
        #     rnn_state,
        # )
        jax.debug.print("call within ScannedRNN")
        # jax.debug.print(carry[0, :5])
        # print(resets.shape)
        new_rnn_state, y = nn.GRUCell(features=ins.shape[1])(rnn_state, ins)
        return new_rnn_state, y

    @staticmethod
    def initialize_carry(batch_size, hidden_size):
        # Use a dummy key since the default state init fn is just zeros.
        cell = nn.GRUCell(features=hidden_size)
        return cell.initialize_carry(jax.random.PRNGKey(0), (batch_size, hidden_size))


class ActorCriticRNN(nn.Module):
    action_dim: Sequence[int]
    config: Dict

    @nn.compact
    def __call__(self, hidden, x):
        # obs, dones = x
        obs = x
        embedding = nn.Dense(
            self.config["FC_DIM_SIZE"], kernel_init=orthogonal(np.sqrt(2)), bias_init=constant(0.0)
        )(obs)
        embedding = nn.relu(embedding)

        # rnn_in = (embedding, dones)
        rnn_in = embedding
        print("call within ActorCriticRNN")
        print(hidden.shape)
        print(rnn_in.shape)
        hidden, embedding = ScannedRNN()(hidden, rnn_in)

        actor_mean = nn.Dense(self.config["GRU_HIDDEN_DIM"], kernel_init=orthogonal(2), bias_init=constant(0.0))(
            embedding
        )
        actor_mean = nn.relu(actor_mean)
        actor_mean = nn.Dense(
            self.action_dim, kernel_init=orthogonal(0.01), bias_init=constant(0.0)
        )(actor_mean)        

        pi = distrax.Categorical(logits=actor_mean)

        critic = nn.Dense(self.config["FC_DIM_SIZE"], kernel_init=orthogonal(2), bias_init=constant(0.0))(
            embedding
        )
        critic = nn.relu(critic)
        critic = nn.Dense(1, kernel_init=orthogonal(1.0), bias_init=constant(0.0))(
            critic
        )

        return hidden, pi, jnp.squeeze(critic, axis=-1)

In [3]:
config = {
    "FC_DIM_SIZE": 128,
    "GRU_HIDDEN_DIM": 128,
    "NUM_ENVS": 16,
}

network = ActorCriticRNN(4, config=config)

In [4]:
rng = jax.random.PRNGKey(0)
rng, _rng = jax.random.split(rng)
# init_x[0]: 1 x num_envs x flattened_obs
# init_x[1]: 1 x num_envs
# init_x = (
#     jnp.zeros(
#         (1, config["NUM_ENVS"], spaces.Box(0, 1, (5 * 5 * 4,)).shape[0])
#     ),
#     jnp.zeros((1, config["NUM_ENVS"])),
# )

init_x = jax.random.uniform(_rng,
    shape=(5, config["NUM_ENVS"], spaces.Box(0, 1, (5 * 5 * 4,)).shape[0])
)

rng, _rng = jax.random.split(rng)
# init_hstate: num_envs x hidden_dim
init_hstate = ScannedRNN.initialize_carry(config["NUM_ENVS"], config["GRU_HIDDEN_DIM"])
network_params = network.init(_rng, init_hstate, init_x)

call within ActorCriticRNN
(16, 128)
(5, 16, 128)
call within ScannedRNN
call within ScannedRNN
call within ScannedRNN
call within ScannedRNN
call within ScannedRNN
call within ScannedRNN


In [5]:
network_params["params"].keys()

dict_keys(['Dense_0', 'ScannedRNN_0', 'Dense_1', 'Dense_2', 'Dense_3', 'Dense_4'])

In [37]:
hstate, pi, value = network.apply(network_params, init_hstate, init_x)

call within ActorCriticRNN
(16, 128)
(5, 16, 128)
call within ScannedRNN
call within ScannedRNN
call within ScannedRNN
call within ScannedRNN
call within ScannedRNN
call within ScannedRNN


In [10]:
hstate.shape

(16, 128)